# TERRA-X: Functional Planetary Analog Discovery Engine
### Reproducible, Uncertainty-Aware Geospatial Architecture for Moon & Mars Exploration
**NASA Space Apps Challenge 2026**

---

## 1. Executive Summary & Non-Negotiables
TERRA-X is **NOT** a visual-similarity map. Visual likeness between Earth and planetary surfaces is often dangerously deceptive (e.g. wet quartz sand or rounded aeolian dunes resembling lunar regolith visually but exhibiting completely invalid bearing capacities and slip dynamics).

Instead, TERRA-X evaluates **each H3 hexagonal cell (r5/r6) as an engineering TEST ARTICLE for a named exploration FUNCTION**:
1. **`MOBILITY`**: Rover traction, sinkage $z/D$, obstacle clearance $h_{\text{rms}}/D$, and slip $\tan\theta/\tan\phi$.
2. **`CONSTRUCTION_ISRU`**: Excavation cutting forces $F/(\gamma d^3)$, regolith sintering, and bearing capacity $p/(\gamma B)$.
3. **`LANDING_PAD_PSI`**: Rocket plume-surface interaction (PSI), supersonic scour, and crater blast ejecta.
4. **`HUMAN_OPS_THERMAL`**: Diurnal thermal swings $\Delta T / T_{\text{mean}}$ and radiative heat sink dynamics.
5. **`PNT_DENIED_NAV`**: Optical terrain-relative navigation (TRN) in deep shadows and grazing illumination.
6. **`ASTROBIO`**: Extreme desiccation, cryptoendolithic preservation, and water activity limits.

### Core Scoring Law
$$\text{AFI}_f = M_f \times (1 - C_f) \times T_f$$
- $M_f$: Confidence-weighted Wasserstein-1 distance match with sensor uncertainty $\tau$.
- $C_f$: Function-dependent Earth-added confounder severity (moisture, vegetation, rounded grains, atmosphere, biology).
- $T_f$: Transferable physics fraction.
- Missing data strictly yields **`UNCONSTRAINED`** (never imputed!).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import numpy as np
import pandas as pd
from targets.registry import TargetRegistry
from features.dimensionless import DimensionlessPhysicsEngine, G_EARTH, G_MOON, G_MARS
from features.confounders import ConfounderEngine
from engine.scoring import ScoringEngine, FidelityStatus
from engine.pipeline import TerraXPipeline
from harmonize.h3_grid import H3Grid

print("TERRA-X Planetary Surface Systems Engine Loaded Successfully.")

## 2. Gravity Bridges & Dimensionless Scaling Laws
A foundational physical rule of TERRA-X is: **Never claim $1g == 1/6g$**.

Granular soil mechanics are governed by effective unit weight $\gamma = \rho g$:
- Earth: $\gamma_{\text{earth}} = 1520 \times 9.81 \approx 14,911\,\text{N/m}^3$
- Moon: $\gamma_{\text{moon}} = 1520 \times 1.62 \approx 2,462\,\text{N/m}^3$
- Load Scale Bridge: $\frac{\gamma_{\text{earth}}}{\gamma_{\text{moon}}} \approx 6.05 - 6.37\times$!

Let us verify the dimensionless groups across targets:

In [ ]:
bridge_moon = DimensionlessPhysicsEngine.get_gravity_bridge("MOON", soil_density_kg_m3=1520.0)
bridge_mars = DimensionlessPhysicsEngine.get_gravity_bridge("MARS", soil_density_kg_m3=1450.0)

print(f"[MOON] Unit Weight: {bridge_moon['gamma_target_N_m3']:.1f} N/m3 | Gravity Load Scale: x{bridge_moon['load_scale_factor']:.2f}")
print(f"[MARS] Unit Weight: {bridge_mars['gamma_target_N_m3']:.1f} N/m3 | Gravity Load Scale: x{bridge_mars['load_scale_factor']:.2f}")

# Compute Dimensionless Groups for Shackleton Crater analog test
groups = DimensionlessPhysicsEngine.compute_all_groups(
    slope_deg=14.5,
    friction_angle_deg=38.0,
    h_rms_m=0.14,
    wheel_diameter_m=0.50,
    contact_pressure_kpa=25.0,
    footprint_width_m=0.30,
    target_body="MOON"
)

for k, v in groups.to_dict().items():
    print(f"  {k:30s}: {v}")

## 3. Confounder Severity & Biological Sign Flip
Earth adds environmental confounders that do not exist in planetary exploration:
- Moisture (capillary cohesion in vacuum regolith)
- Vegetation (root cohesion, radar attenuation)
- Atmosphere (aerodynamic drag, convective cooling vs pure radiation)
- Rounded grains (fluvial/aeolian rounding drastically under-predicts regolith shear)

**Sign Inversion Rule:** In engineering functions (`MOBILITY`, `CONSTRUCTION`), biological organic contamination is a confounder. In `ASTROBIO`, biological extremophile presence is the **TARGET SIGNAL**!

In [ ]:
conf_mobility = ConfounderEngine.evaluate(
    function_profile="MOBILITY",
    soil_moisture_wtd=0.012,
    vegetation_ndvi=0.02,
    grain_wadell_roundness=0.25,
    biological_activity_index=0.20
)

conf_astrobio = ConfounderEngine.evaluate(
    function_profile="ASTROBIO",
    soil_moisture_wtd=0.012,
    vegetation_ndvi=0.02,
    grain_wadell_roundness=0.25,
    biological_activity_index=0.20
)

print(f"Mobility Biological Penalty : {conf_mobility.biological_penalty:+.4f} (CONFOUNDER)")
print(f"Astrobio Biological Penalty : {conf_astrobio.biological_penalty:+.4f} (INVERTED TO SIGNAL!)")

## 4. End-to-End Pipeline Execution: Moon South Pole (Artemis III/IV)
Now let us run the 7-stage autonomous discovery pipeline for the **Moon South Pole (Shackleton Rim)** targeting **`MOBILITY`** and **`CONSTRUCTION_ISRU`**.

In [ ]:
pipeline = TerraXPipeline()

# Run discovery with 100-clone Monte Carlo simulation
discovery_results = pipeline.run_discovery_pipeline(
    target_id="MOON_SOUTH_POLE_SHACKLETON",
    function_profile="MOBILITY",
    wheel_diameter_m=0.50,
    contact_pressure_kpa=25.0,
    top_k=10,
    enforce_mmr=True
)

df_ranked = pd.DataFrame([
    {
        "Rank (p50)": s["monte_carlo_rank"]["rank_p50"],
        "Site Name": s["site_name"],
        "Country": s["country"],
        "H3 Cell": s["cell_id_h3_r6"],
        "AFI Score": s["scores"]["afi_p50"],
        "Match (M)": s["scores"]["M_match"],
        "Confounder (C)": s["scores"]["C_confounder"],
        "Transferable (T)": s["scores"]["T_transferable"],
        "Rank p10-p90": f"#{s['monte_carlo_rank']['rank_p10']} - #{s['monte_carlo_rank']['rank_p90']}",
        "P(top10)": f"{s['monte_carlo_rank']['p_top10']*100:.0f}%",
        "Novelty": s["novelty"]["status"]
    }
    for s in discovery_results["ranked_sites"]
])

df_ranked

## 5. Site Passport & Fidelity Ledger Inspection
Let us inspect the top-ranked site passport to examine its **Fidelity Ledger** (`STRONG`, `PARTIAL`, `BREAKS`, `UNCONSTRAINED`) and verify that hard breaks are strictly surfaced.

In [ ]:
top_passport = discovery_results["ranked_sites"][0]
print(f"SITE PASSPORT: {top_passport['site_name']} ({top_passport['site_id']})")
print(f"Target Body   : {top_passport['planetary_target']['body']} - {top_passport['planetary_target']['target_name']}")
print(f"H3 Cell ID    : {top_passport['cell_id_h3_r6']}")
print("\nMANDATORY HARD BREAKS:")
for hb in top_passport["hard_breaks"]:
    print(f"  [!] {hb}")

print("\nFIDELITY LEDGER:")
df_ledger = pd.DataFrame(top_passport["fidelity_ledger"])
df_ledger[["subsystem_name", "status", "earth_value", "target_value", "transferability_fraction", "notes"]]

## 6. Transferable Test Recipe Tied to NASA Tech Gaps
The test recipe provides actionable, verifiable instructions for conducting flight hardware tests on the candidate test article, including **gravity compensation rig parameters** and **regolith simulant conditioning**.

In [ ]:
recipe = top_passport["test_recipe"]
import json
print(json.dumps(recipe, indent=2))